In [4]:
import argparse
import csv
import json
import logging
import re
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

# SETTINGS
SEARCH_BASE_URL = "https://wuzzuf.net/saudi/search/jobs?filters%5Bcountry%5D%5B0%5D=Saudi%20Arabia"
SEARCH_PARAMS = {"q": "", "a": "spbg"}   # extra query params for the search endpoint
JSON_OUTPUT = "../data/raw/wuzzuf_jobs.json"
CSV_OUTPUT = "../data/raw/wuzzuf_jobs.csv"

REQUEST_TIMEOUT = 20
MAX_RETRIES = 3
RETRY_BACKOFF_SECONDS = 5
LISTING_PAGE_DELAY = 3.0                  # be polite between search pages
DETAIL_PAGE_DELAY = 0.5                   # small stagger between job-detail fetches
DEFAULT_WORKERS = 4                       # concurrent job-detail fetches

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9,ar;q=0.5",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

STORE_STATE_PATTERN = re.compile(
    r"Wuzzuf\.initialStoreState\s*=\s*(\{.*?\})\s*;\s*\n",
    re.DOTALL,
)
STORE_STATE_FALLBACK_PATTERN = re.compile(
    r"Wuzzuf\.initialStoreState\s*=\s*(\{.*\})",
    re.DOTALL,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("wuzzuf")


def make_session():
    session = requests.Session()
    session.headers.update(HEADERS)
    return session


def fetch_html(session, url, params=None):
    """GET a URL with retries + backoff. Returns response text, or
    None if every attempt failed."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = session.get(url, params=params, timeout=REQUEST_TIMEOUT)
            if response.status_code != 200:
                log.warning("Non-200 (%s) for %s (attempt %d/%d)",
                            response.status_code, response.url, attempt, MAX_RETRIES)
                if response.status_code in (403, 429):
                    time.sleep(RETRY_BACKOFF_SECONDS * attempt)
                    continue
                return None
            response.encoding = response.apparent_encoding or "utf-8"
            return response.text
        except requests.exceptions.RequestException as e:
            log.warning("Request failed (%s) attempt %d/%d: %s", url, attempt, MAX_RETRIES, e)
            time.sleep(RETRY_BACKOFF_SECONDS * attempt)
    return None
 
# STAGE 1 - COLLECT JOB URLS FROM SEARCH RESULT PAGES

def collect_job_urls(session, max_pages=None):
    """Paginate through the Wuzzuf search results and collect every
    unique job-detail URL. Stops when a page yields no new job links
    or `max_pages` is reached."""
    job_urls = []
    seen = set()
    page = 0

    while True:
        if max_pages is not None and page >= max_pages:
            log.info("Reached max_pages=%d, stopping URL collection.", max_pages)
            break

        params = dict(SEARCH_PARAMS)
        if page > 0:
            params["start"] = page

        log.info("Scraping search page %d...", page + 1)
        html = fetch_html(session, SEARCH_BASE_URL, params=params)
        if html is None:
            log.error("Search page %d failed after retries. Stopping.", page + 1)
            break

        soup = BeautifulSoup(html, "html.parser")
        page_urls = []
        for a in soup.find_all("a", href=True):
            href = a["href"]
            if "/jobs/p/" in href:
                page_urls.append(urljoin("https://wuzzuf.net", href))

        new_urls = [u for u in dict.fromkeys(page_urls) if u not in seen]
        log.info("  found %d links (%d new)", len(page_urls), len(new_urls))

        if not new_urls:
            log.info("No new jobs on this page. Stopping pagination.")
            break

        for u in new_urls:
            seen.add(u)
            job_urls.append(u)

        log.info("  running total: %d unique job URLs", len(job_urls))
        page += 1
        time.sleep(LISTING_PAGE_DELAY)

    return job_urls
 
# STAGE 2 - PARSE ONE JOB PAGE

def extract_store_state(html):
    """Wuzzuf embeds the full job payload server-side as JSON assigned
    to Wuzzuf.initialStoreState inside an inline <script> tag. Pull
    that blob out directly and parse it."""
    match = STORE_STATE_PATTERN.search(html)
    if not match:
        match = STORE_STATE_FALLBACK_PATTERN.search(html)
    if not match:
        raise ValueError("Could not find 'Wuzzuf.initialStoreState' in the page HTML.")

    raw_json = match.group(1)
    try:
        return json.loads(raw_json)
    except json.JSONDecodeError:
        return json.loads(_trim_to_balanced_json(raw_json))


def _trim_to_balanced_json(text):
    start = text.find("{")
    if start == -1:
        raise ValueError("No opening brace found in store-state blob.")
    depth = 0
    in_string = False
    escape = False
    for i in range(start, len(text)):
        char = text[i]
        if in_string:
            if escape:
                escape = False
            elif char == "\\":
                escape = True
            elif char == '"':
                in_string = False
            continue
        if char == '"':
            in_string = True
        elif char == "{":
            depth += 1
        elif char == "}":
            depth -= 1
            if depth == 0:
                return text[start:i + 1]
    raise ValueError("Could not find a balanced closing brace for the JSON blob.")


def localized(node, field, fallback_field=None):
    """Pull the English translation of `field` off a lookup object
    shaped like {"name": "...", "translations": {"name": {"ar": "...",
    "en": "..."}}}. Falls back to the raw field value, then to
    `fallback_field` if given."""
    if not isinstance(node, dict):
        return None
    translations = node.get("translations") or {}
    field_translations = translations.get(field)
    if isinstance(field_translations, dict) and field_translations.get("en"):
        return field_translations["en"]
    if node.get(field):
        return node[field]
    if fallback_field and node.get(fallback_field):
        return node[fallback_field]
    return None


def html_to_bullets(html_text):
    """Convert a job description/requirements HTML blob into a flat
    list of plain-text bullet lines."""
    if not html_text:
        return []
    soup = BeautifulSoup(html_text, "html.parser")
    items = soup.find_all("li")
    if items:
        bullets = [li.get_text(" ", strip=True) for li in items if li.get_text(strip=True)]
        if bullets:
            return bullets
    text = soup.get_text("\n", strip=True)
    return [line for line in text.split("\n") if line]


def format_experience(years_obj):
    if not isinstance(years_obj, dict):
        return None
    lo, hi = years_obj.get("min"), years_obj.get("max")
    if lo is None and hi is None:
        return None
    if lo is None:
        return f"0 - {hi} years"
    if hi is None:
        return f"{lo}+ years"
    if lo == hi:
        return f"{lo} years"
    return f"{lo} - {hi} years"


def format_salary(salary_obj):
    if not isinstance(salary_obj, dict):
        return None
    if not salary_obj.get("isPaid", True):
        return "Unpaid"
    lo, hi = salary_obj.get("min"), salary_obj.get("max")
    currency = salary_obj.get("currency") or {}
    currency_code = currency.get("code") or localized(currency, "name")
    period = salary_obj.get("period") or {}
    period_text = localized(period, "period") or period.get("name")
    extra = salary_obj.get("additionalDetails")

    if lo is None and hi is None:
        base = "Confidential"
    elif lo is not None and hi is not None:
        base = f"{lo:,} - {hi:,}"
    else:
        base = f"{lo or hi:,}"

    parts = [base]
    if currency_code and base != "Confidential":
        parts.append(currency_code)
    if period_text and base != "Confidential":
        parts.append(period_text)
    text = " ".join(parts)
    if extra:
        text = f"{text} ({extra})"
    return text


def find_job_id_by_slug(store, url):
    """Match the job entity to this URL via its slug (the part after
    '/p/' in the URL) rather than assuming it's the only job entity in
    the normalized store - 'similar jobs' entities live there too."""
    slug_match = re.search(r"/p/([^/?]+)", url)
    target_slug = slug_match.group(1) if slug_match else None

    job_collection = store.get("entities", {}).get("job", {}).get("collection", {})
    if target_slug:
        for job_id, job_entity in job_collection.items():
            attrs = job_entity.get("attributes", {})
            if attrs.get("slug") == target_slug:
                return job_id, job_entity

    similar_jobs = store.get("jobPage", {}).get("similarJobs", {})
    if similar_jobs:
        main_job_id = next(iter(similar_jobs))
        job_entity = job_collection.get(main_job_id)
        if job_entity:
            return main_job_id, job_entity

    if job_collection:
        job_id = next(iter(job_collection))
        return job_id, job_collection[job_id]

    return None, None


def get_company(store, job_entity):
    company_ref = job_entity.get("relationships", {}).get("company", {}).get("data", {})
    if not company_ref:
        return None
    company_id = company_ref.get("id")
    return store.get("entities", {}).get("company", {}).get("collection", {}).get(company_id)


def normalize_job(job_entity, company_entity, job_url):
    attrs = job_entity.get("attributes", {})
    user_translations = attrs.get("userContentTranslations", {})

    title = (user_translations.get("title") or {}).get("en") or attrs.get("title")
    company_attrs = (company_entity or {}).get("attributes", {})

    location = attrs.get("location") or {}
    city_obj = location.get("city") or {}
    country_obj = location.get("country") or {}
    area_obj = location.get("area") or {}
    location_parts = [
        localized(area_obj, "name"),
        localized(city_obj, "name"),
        localized(country_obj, "name"),
    ]
    location_text = ", ".join(p for p in location_parts if p) or None

    work_types = attrs.get("workTypes") or []
    job_type = ", ".join(
        filter(None, (localized(wt, "displayedName") for wt in work_types))
    ) or None

    workplace = attrs.get("workplaceArrangement") or {}
    work_mode = localized(workplace, "displayedName")

    career_level = attrs.get("careerLevel") or {}
    career_level_text = localized(career_level, "displayName") or localized(career_level, "name")

    education_level = (attrs.get("candidatePreferences") or {}).get("educationLevel") or {}
    qualification = localized(education_level, "name")

    work_roles = attrs.get("workRoles") or []
    job_categories_list = [r for r in (localized(role, "name") for role in work_roles) if r]

    keywords = attrs.get("keywords") or []
    skills_and_tools = [k.get("name") for k in keywords if k.get("name")]

    return {
        "job_title": title,
        "company": company_attrs.get("name"),
        "location": location_text,
        "job_type": job_type,
        "work_mode": work_mode,
        "posted": attrs.get("postedAt"),
        "job_url": job_url,
        "experience_needed": format_experience(attrs.get("workExperienceYears")),
        "career_level": career_level_text,
        "qualification": qualification,
        "salary": format_salary(attrs.get("salary")),
        "job_categories_list": job_categories_list,
        "skills_and_tools": skills_and_tools,
        "job_description": html_to_bullets(attrs.get("description")),
        "job_requirements": html_to_bullets(attrs.get("requirements")),
    }


def scrape_job_detail(session, url):
    """Fetch + parse a single job page. Returns a normalized dict, or
    None on failure (never raises - callers run this in a thread pool)."""
    try:
        html = fetch_html(session, url)
        if html is None:
            log.error("Failed to download job page: %s", url)
            return None

        store_state = extract_store_state(html)
        job_id, job_entity = find_job_id_by_slug(store_state, url)
        if not job_entity:
            log.error("Could not locate job entity in store state: %s", url)
            return None

        company_entity = get_company(store_state, job_entity)
        return normalize_job(job_entity, company_entity, url)

    except Exception as e:
        log.error("Error parsing %s: %s: %s", url, type(e).__name__, e)
        return None

# SAVE HELPERS
def save_json(records, filename):
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)



def _stringify_csv_value(value):
    if value is None:
        return ""
    if isinstance(value, list):
        return " | ".join(
            item if isinstance(item, str) else json.dumps(item, ensure_ascii=False)
            for item in value
        )
    if isinstance(value, dict):
        return json.dumps(value, ensure_ascii=False)
    return value


def save_csv(records, filename):
    if not records:
        return
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    fieldnames = list(records[0].keys())
    for r in records:
        for k in r.keys():
            if k not in fieldnames:
                fieldnames.append(k)

    with open(filename, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for r in records:
            writer.writerow({k: _stringify_csv_value(v) for k, v in r.items()})



def save_urls(urls, filename):
    dirname = os.path.dirname(filename)
    if dirname:
        os.makedirs(dirname, exist_ok=True)
    with open(filename, "w", encoding="utf-8") as f:
        f.write("\n".join(urls))

 
# MAIN
def run_scraper(max_pages=None, max_jobs=None, workers=DEFAULT_WORKERS):
    """Core pipeline, callable directly from a notebook cell —
    e.g. run_scraper(max_pages=2, max_jobs=5) — with no argparse/
    sys.argv involved at all."""
    session = make_session()

    # ---- Stage 1: collect job URLs ----
    log.info("=" * 70)
    log.info("STAGE 1: Collecting job URLs from search results")
    log.info("=" * 70)
    job_urls = collect_job_urls(session, max_pages=max_pages)

    if max_jobs is not None:
        job_urls = job_urls[:max_jobs]
        log.info("Limiting to first %d jobs for detail scraping.", len(job_urls))

    if not job_urls:
        log.warning("No job URLs found - nothing to scrape. Exiting.")
        return [], []

    # ---- Stage 2: scrape each job's details ----
    log.info("=" * 70)
    log.info("STAGE 2: Scraping details for %d jobs (workers=%d)", len(job_urls), workers)
    log.info("=" * 70)

    records = []
    failed_urls = []

    def worker(url):
        # Each thread uses its own session to avoid connection-pool contention.
        thread_session = make_session()
        result = scrape_job_detail(thread_session, url)
        time.sleep(DETAIL_PAGE_DELAY)
        return url, result

    with ThreadPoolExecutor(max_workers=workers) as executor:
        futures = {executor.submit(worker, url): url for url in job_urls}
        done = 0
        for future in as_completed(futures):
            url, result = future.result()
            done += 1
            if result:
                records.append(result)
                log.info("[%d/%d] OK   - %s", done, len(job_urls), result.get("job_title") or url)
            else:
                failed_urls.append(url)
                log.info("[%d/%d] FAIL - %s", done, len(job_urls), url)

            # Save progress periodically so a crash doesn't lose everything.
            if done % 10 == 0 or done == len(job_urls):
                save_json(records, JSON_OUTPUT)
                save_csv(records, CSV_OUTPUT)

    save_json(records, JSON_OUTPUT)
    save_csv(records, CSV_OUTPUT)

    log.info("=" * 70)
    log.info("DONE")
    log.info("=" * 70)
    log.info("Scraped:  %d / %d jobs", len(records), len(job_urls))
    log.info("Failed:   %d jobs", len(failed_urls))
    log.info("JSON out: %s", JSON_OUTPUT)
    log.info("CSV out:  %s", CSV_OUTPUT)
    if failed_urls:
        failed_file = "wuzzuf_failed_urls.txt"
        save_urls(failed_urls, failed_file)
        log.info("Failed URLs saved to: %s", failed_file)

    return records, failed_urls


def main():
    """Entry point for running as a plain script: `python wuzzuf_scraper.py ...`"""
    parser = argparse.ArgumentParser(description="Scrape Wuzzuf job listings + details.")
    parser.add_argument("--max-pages", type=int, default=None,
                         help="Stop after this many search-result pages (default: no limit).")
    parser.add_argument("--max-jobs", type=int, default=None,
                         help="Only scrape details for this many jobs (useful for a quick test).")
    parser.add_argument("--workers", type=int, default=DEFAULT_WORKERS,
                         help=f"Concurrent job-detail fetch threads (default: {DEFAULT_WORKERS}).")
    # parse_known_args() ignores extra args a notebook kernel injects (e.g. --f=...kernel.json)
    args, _unknown = parser.parse_known_args()
    run_scraper(max_pages=args.max_pages, max_jobs=args.max_jobs, workers=args.workers)

if __name__ == "__main__":
    main()

15:55:22 [INFO] ======================================================================
15:55:22 [INFO] STAGE 1: Collecting job URLs from search results
15:55:22 [INFO] ======================================================================
15:55:22 [INFO] Scraping search page 1...
15:55:24 [INFO]   found 15 links (15 new)
15:55:24 [INFO]   running total: 15 unique job URLs
15:55:27 [INFO] Scraping search page 2...
15:55:28 [INFO]   found 15 links (15 new)
15:55:28 [INFO]   running total: 30 unique job URLs
15:55:31 [INFO] Scraping search page 3...
15:55:33 [INFO]   found 15 links (15 new)
15:55:33 [INFO]   running total: 45 unique job URLs
15:55:36 [INFO] Scraping search page 4...
15:55:39 [INFO]   found 15 links (15 new)
15:55:39 [INFO]   running total: 60 unique job URLs
15:55:42 [INFO] Scraping search page 5...
15:55:43 [INFO]   found 15 links (15 new)
15:55:43 [INFO]   running total: 75 unique job URLs
15:55:46 [INFO] Scraping search page 6...
15:55:48 [INFO]   found 13 links (13 ne